In [ ]:
import requests
import pandas as pd
import time

# Replace with your own Gaode API Key
API_KEY = ""


KEYWORDS = ["data center", "IDC", "Internet Data Center", "big data center", "cloud computing center", "machine room"]


PROVINCES = {
 "Tianjin": "120000", "Shanxi": "140000", "Neimenggu (Inner Mongolia)": "150000",    
 "Liaoning": "210000", "Jilin": "220000", "Heilongjiang": "230000",    
 "Shanghai": "310000", "Jiangsu": "320000", "Zhejiang": "330000", "Anhui": "340000",     
 "Fujian": "350000", "Shandong": "370000",    
 "Hubei": "420000", "Hunan": "430000", "Guangxi": "450000", "Hainan": "460000",    
 "Chongqing": "500000", "Sichuan": "510000", "Guizhou": "520000", "Yunnan": "530000", "Xizang (Tibet)": "540000",    
 "Gansu": "620000", "Qinghai": "630000", "Ningxia": "640000", "Xinjiang": "650000"
}

results = []


for prov, adcode in PROVINCES.items():
    print(f" {prov} ...")
    for keyword in KEYWORDS:
        page = 1
        while True:
            url = (
                f"https://restapi.amap.com/v3/place/text?"
                f"key={API_KEY}&keywords={keyword}&city={adcode}"
                f"&output=json&page={page}&offset=20"
            )
            try:
                r = requests.get(url, timeout=10).json()
                pois = r.get("pois", [])
            except Exception as e:
                print(f"：{e}")
                break

            if not pois: 
                break

            for p in pois:
                results.append({
                    "province": prov,
                    "id": p.get("id", None),
                    "name": p.get("name", None),
                    "address": p.get("address", None),
                    "location": p.get("location", None),  
                    "adcode": p.get("adcode", None),
                    "cityname": p.get("cityname", None),
                })
            page += 1
            time.sleep(0.2)  


df = pd.DataFrame(results)
df.drop_duplicates(subset=["id"], inplace=True, ignore_index=True)

save_path = "china_datacenter_POI.csv"
df.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"The nationwide data center POI crawling has been completed, with a total of {len(df)} entries, which have been saved to {save_path}")
